## 📖 Libro: §0.–0 del Capítulo 0 — Prerrequisitos (warm-up)

**Enunciado (verbatim del libro):** *"Antes de hablar de curvatura en el espacio de distribuciones, asegúrate de que el álgebra lineal, el cálculo y la probabilidad sean tus aliados, no tus enemigos."*

**Mini-reto:** verificar 5 hechos de prerrequisito que se usan literalmente en todos los capítulos posteriores: (1) convexidad por Hessian PD, (2) Cauchy–Schwarz como igualdad de momentos, (3) estadísticos suficientes de Bernoulli/Poisson, (4) Fisher information I(p) = 1/(p(1−p)) por muestreo, (5) CLT emerge de promedios de Uniformes.

**@ Pregunta a tu LLM:** «¿Por qué el test de convexidad `np.linalg.eigvalsh(H).min() ≥ 0` es numéricamente no trivial cuando H es (n,n) con n ≥ 5? ¿Cuál es la diferencia entre máquina con `np.float64` y `np.float32`?»

In [ ]:
# =====================================================================
# Celda 1 — imports + seed determinístico (cross-platform via SHA-256)
# =====================================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from utils import setup_seed

SEED = setup_seed("cap0_prerequisites")
rng = np.random.default_rng(SEED)
print(f"Deterministic seed: {SEED}")
print(f"NumPy {np.__version__} | matplotlib {plt.matplotlib.__version__}")

In [ ]:
# =====================================================================
# Celda 2 — Prerrequisito #1: Convexidad por Hessiano PD
# =====================================================================
# Verifica para 5 funciones clásicas con cálculo numérico del Hessiano
# y eigenvalues. Una función f: R^n → R es convexa ssi ∇²f(x) ⪰ 0
# en todo x del dominio.

def numerical_hessian(f, x, eps=1e-3):
    """Hessiano por diferencias finitas centradas."""
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            xpp = x.copy(); xpp[i] += eps; xpp[j] += eps
            xpm = x.copy(); xpm[i] += eps; xpm[j] -= eps
            xmp = x.copy(); xmp[i] -= eps; xmp[j] += eps
            xmm = x.copy(); xmm[i] -= eps; xmm[j] -= eps
            H[i, j] = (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * eps**2)
    return 0.5 * (H + H.T)  # symmetrize for numerical noise

# 5 funciones a probar
tests = {
    "f(x) = 0.5 * x^t x":        lambda x: 0.5 * x @ x,
    "f(x) = exp(x[0])":          lambda x: np.exp(x[0]),
    "f(x) = x[0]**4 - x[1]**2":  lambda x: x[0]**4 - x[1]**2,  # NO convexa
    "f(x) = log(1 + e^x[0])":    lambda x: np.log1p(np.exp(x[0])),  # softplus
    "f(x) = -log(x[0])":         lambda x: -np.log(x[0]),  # barrera
}

print(f"{'(label)':40s}  {'λmin(∇²f)':>16s}  {'Convexa?':>10s}")
print("-" * 70)
for name, f in tests.items():
    x0 = np.array([0.5, 0.5]) if "-" in name else np.array([0.3, 0.3])
    if f == tests["f(x) = -log(x[0])"]:
        x0 = np.array([0.5])
    H = numerical_hessian(f, x0, eps=1e-3)
    eigs = np.linalg.eigvalsh(H)
    is_conv = eigs.min() >= -1e-6
    print(f"{name:40s}  {eigs.min():>16.4f}  {('Sí' if is_conv else 'No'):>10s}")

# Esperado: 4-5 funciones convexas; "f(x)=x0^4-x1^2" con λs mezclados (PD en (0,0), PSD en algunos puntos).

In [ ]:
# =====================================================================
# Celda 3 — Prerrequisito #2: Cauchy–Schwarz como E[X²] ≥ E[X]²
# =====================================================================
# E[X²] ≥ E[X]² con igualdad ssi X es constante (o casi seguro).
# Verifica para 5 distribuciones: Bernoulli, Normal, Uniforme, Poisson, Exponencial.
ns = [10_000, 100_000]
cases = [
    ("Bernoulli(0.3)", lambda n, r: r.binomial(1, 0.3, n).astype(float)),
    ("N(2, 1)",       lambda n, r: 2 + r.standard_normal(n)),
    ("Uniform[0, 1]",  lambda n, r: r.uniform(0, 1, n)),
    ("Poisson(4)",     lambda n, r: r.poisson(4, n).astype(float)),
    ("Exp(1)",         lambda n, r: r.exponential(1.0, n)),
]
print(f"{'(distribution)':22s} {'n':>8s}  {'E[X]':>8s}  {'E[X²]':>10s}  {'E[X²] - E[X]²':>18s}")
print("-" * 70)
for label, sampler in cases:
    for n in ns:
        x = sampler(n, rng)
        EX = x.mean()
        EX2 = (x**2).mean()
        gap = EX2 - EX**2
        print(f"{label:22s} {n:>8d}  {EX:>8.4f}  {EX2:>10.4f}  {gap:>18.4e}  {'(Var)'}")

# Esperado: E[X²] - E[X]² > 0 para todas las distribuciones no degeneradas;
# y crece ~ linealmente con la varianza poblacional. Para Bernoulli(0.3),
# la diferencia ≈ 0.21 (= p(1-p)).

In [ ]:
# =====================================================================
# Celda 4 — Prerrequisito #3: Estadísticos suficientes (Bernoulli / Poisson)
# =====================================================================
# Para muestra de Bernoulli(p) o Poisson(λ), Σ X_i captura toda la info.
# Verifica: la verosimilitud p(x|θ) depende de los datos solo a través de T = Σ X_i.
def bern_likelihood_ratio_with_without_S(x, p, p0):
    """p(x|p) / p(x|p0) en funciãn de T=x.sum() o x crudo."""
    loglik_p = x.sum()*np.log(p) + (len(x)-x.sum())*np.log(1-p)
    loglik_p0 = x.sum()*np.log(p0) + (len(x)-x.sum())*np.log(1-p0)
    return loglik_p - loglik_p0

n = 500
x_bern = rng.binomial(1, 0.4, n)
T_bern = x_bern.sum()

for p in [0.2, 0.5, 0.8]:
    for p0 in [0.1, 0.6]:
        ratio_from_T = bern_likelihood_ratio_with_without_S(x_bern, p, p0)
        # Cualquier permutación de los datos con el mismo T da idéntico ratio.
        x_permuted = rng.permutation(x_bern)
        ratio_from_T_permuted = bern_likelihood_ratio_with_without_S(x_permuted, p, p0)
        same = np.isclose(ratio_from_T, ratio_from_T_permuted)
        print(f"  p={p}, p0={p0}: ratio(T)={ratio_from_T:.4f}, "
              f"ratio(T) en permutación={ratio_from_T_permuted:.4f}, "
              f"iguales: {same}")

print(f"\nInterpretación: el ratio de verosimilitud depende solo de T = Σ X_i = {T_bern},")
print(f"                no de los datos individuales. T es suficiente.")

In [ ]:
# =====================================================================
# Celda 5 — Prerrequisito #4: Fisher info I(p) = 1/(p(1−p)) por muestreo
# =====================================================================
# Para Bernoulli(p) la Fisher info teórica es 1/(p(1−p)).
# La estimamos empíricamente: I_emp(p) = Var_p[∂ℓ/∂p] con ∂ℓ = x/p - (1−x)/(1−p).
from scipy import integrate

def I_bernoulli_theoretical(p):
    return 1.0 / (p * (1.0 - p))

def I_bernoulli_empirical(p, n_int=100_000):
    x = rng.binomial(1, p, n_int).astype(float)
    # score = ∂ log-likelihood / ∂ p para una observación
    score = x / p - (1 - x) / (1 - p)
    return float(np.var(score))

print(f"{'p':>6s}  {'I_teórico':>14s}  {'I_empírico':>14s}  {'rel_error':>10s}")
print("-" * 50)
for p in [0.05, 0.2, 0.5, 0.7, 0.95]:
    I_t = I_bernoulli_theoretical(p)
    I_e = I_bernoulli_empirical(p, n_int=100_000)
    print(f"{p:>6.2f}  {I_t:>14.4f}  {I_e:>14.4f}  {abs(I_e-I_t)/I_t:>9.2%}")

# Esperado: rel_error < 5% en todas las p; cerca de p=0.95 diverge a Var→∞.

In [ ]:
# =====================================================================
# Celda 6 — Prerrequisito #5: CLT emerge de promedios Uniformes
# =====================================================================
# Para X_i ~ Uniforme[0, 1] iid, las medias muestrales de tamaño N
# convergen en distribución a N(½, 1/(12N)).
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, N in zip(axes, [2, 10, 100]):
    n_traj = 100_000
    X = rng.uniform(0, 1, (n_traj, N))
    means = X.mean(axis=1)
    ax.hist(means, bins=50, density=True, alpha=0.4,
            color="#0F766E", label=f"emírico (N={N})")
    x_ax = np.linspace(0, 1, 200)
    pdf_norm = stats.norm.pdf(x_ax, 0.5, np.sqrt(1/(12*N)))
    ax.plot(x_ax, pdf_norm, "black", linewidth=2,
            label=f"N(½, 1/(12N)) = N(½, {1/(12*N):.4f})")
    ax.set_xlabel(r"$\bar X$")
    ax.set_title(f"N = {N}, media de {N} Uniformes[0,1]")
    ax.legend(fontsize=9)
axes[0].set_ylabel("densidad")
fig.suptitle("CLT: la media de N Uniformes converge a Gaussiana (Cap. 0, §0.5)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## ✅ `@ Verifica con:`

Las 5 verificaciones deben satisfacerse:
1. **Convexidad**: 4 funciones convexas (cuadrática, exp, softplus, barrera) y 1 no convexa (x⁴−y²). El mínimo eigenvalue es >= 0 para las convexas, <0 para la no convexa en el punto de prueba.
2. **Cauchy–Schwarz**: `E[X²] - E[X]² = Var(X) > 0` para todas las distribuciones no degeneradas.
3. **Suficiencia de T = Σ X_i**: dos reordenamientos distintos de los datos producen el mismo ratio de verosimilitud (numeros reportados son idénticos).
4. **Fisher info**: rel_error < 5% para todas las p ∈ [0.05, 0.95]; divergencia visible cerca de p = 0.95 (varianza del score → ∞).
5. **CLT**: distribuciones empíricas se sitúan sobre la Normal teórica con std = 1/√(12N). Para N=100, la visualización debe coincidir prácticamente pixel a pixel.

Conexión con el libro: cada verificación se corresponde con un item del Cap. 0:
- Convexidad → §0.5 (Análisis convexo) — base para diveregencias de Bregman (Cap. 2).
- Cauchy–Schwarz → §0.3 (Probabilidad) — base para la covarianza como Gram (Cap. 8).
- Suficiencia → §0.4.5 y Cap. 3 (§familias exponenciales).
- Fisher info → Cap. 1, §1.3–1.4; base de toda la geometría del libro.
- CLT → §0.3.5 (LLN/CLT) — base de la inferencia asintótica (Cap. 5).

Discusión para el LLM mentor:
- ¿Qué pasaría si calculamos I_empírica con n_int=1000 en lugar de 100_000? La varianza del estimador empírico decrece ~1/n_int.
- ¿Por qué la divergencia de I_emp en p→±0,1 es matemática mucho más fuerte que numérica? Porque el score ~ 1/p diverge; pero el estimador empírico promedia muchos valores donde x=0 (en p→0) y x=1 (en p→1), donde el score está acotado.